# Détection de deepfakes avec ResNet-18

Classifieur binaire sur les visages extraits de **Celeb-DF v2** : classe `0` = **REAL**, `1` = **FAKE**.

Ce notebook applique un protocole d'évaluation strict :

- split **par vidéo** (contrôlé programmatiquement) et test jamais utilisé pour les choix de modèle ;
- checkpoint sélectionné sur l'**AUC de validation** ;
- **seuil de décision choisi sur la validation**, puis appliqué au test ;
- métriques au niveau **frame** *et* au niveau **vidéo** (les frames d'une vidéo ne sont pas indépendantes) ;
- intervalle de confiance bootstrap et test de McNemar entre les deux modèles ;
- comparaison au CNN Baseline en mode **déterministe** (ses couches d'augmentation sont retirées) ;
- analyse d'erreurs, Grad-CAM et test de robustesse à la recompression JPEG.

> Avant l'exécution : télécharger Celeb-DF v2 conformément à ses conditions, préparer
> `split_par_video.csv`, puis renseigner les chemins ci-dessous (ou les variables
> d'environnement `DEEPFAAKE_*`). Le jeu complet n'est pas inclus dans le dépôt.


In [ ]:
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import v2

SEED = int(os.environ.get("DEEPFAAKE_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_DIR = Path.cwd()
KAGGLE_INPUT = Path("/kaggle/input")


def _env_path(name, default):
    return Path(os.environ[name]) if name in os.environ else default


# --- Dataset ------------------------------------------------------------
if "DEEPFAAKE_DATASET_DIR" in os.environ:
    DATASET_DIR = _env_path("DEEPFAAKE_DATASET_DIR", None)
elif KAGGLE_INPUT.is_dir():
    candidates = sorted(p for p in KAGGLE_INPUT.rglob("Celeb_V2") if p.is_dir())
    DATASET_DIR = candidates[0] if len(candidates) == 1 else Path("/kaggle/input/CelebDF_V2/Celeb_V2")
else:
    DATASET_DIR = PROJECT_DIR / "Celeb_V2"

# --- CSV de split -------------------------------------------------------
if "DEEPFAAKE_SPLIT_CSV" in os.environ:
    SPLIT_CSV = _env_path("DEEPFAAKE_SPLIT_CSV", None)
elif KAGGLE_INPUT.is_dir():
    csv_candidates = sorted(KAGGLE_INPUT.rglob("split_par_video.csv"))
    preferred = [p for p in csv_candidates if (p.parent / "cnn_simple.keras").is_file()]
    SPLIT_CSV = (preferred or csv_candidates or [Path("/kaggle/input/deepfake-v1-livrable/split_par_video.csv")])[0]
else:
    SPLIT_CSV = PROJECT_DIR / "result_etudiant1" / "split_par_video.csv"

# --- Modèle CNN (comparaison facultative) -------------------------------
if "DEEPFAAKE_CNN_MODEL" in os.environ:
    CNN_MODEL_PATH = _env_path("DEEPFAAKE_CNN_MODEL", None)
elif KAGGLE_INPUT.is_dir():
    CNN_MODEL_PATH = SPLIT_CSV.parent / "cnn_simple.keras"
else:
    CNN_MODEL_PATH = PROJECT_DIR / "cnn_simple.keras"

# --- Sorties ------------------------------------------------------------
if "DEEPFAAKE_OUTPUT_DIR" in os.environ:
    OUTPUT_DIR = _env_path("DEEPFAAKE_OUTPUT_DIR", None)
elif KAGGLE_INPUT.is_dir():
    OUTPUT_DIR = Path("/kaggle/working/result_etudiant2/reproduction")
else:
    OUTPUT_DIR = PROJECT_DIR / "result_etudiant2" / "reproduction"

IMG_SIZE = int(os.environ.get("DEEPFAAKE_IMG_SIZE", "128"))
BATCH_SIZE = int(os.environ.get("DEEPFAAKE_BATCH_SIZE", "64"))
# 0 par défaut sous macOS/Windows (portabilité) ; 4 sous Linux pour éviter le goulot I/O.
NUM_WORKERS = int(os.environ.get("DEEPFAAKE_WORKERS", "4" if sys.platform.startswith("linux") else "0"))
EPOCHS_HEAD = int(os.environ.get("DEEPFAAKE_EPOCHS_HEAD", "3"))
EPOCHS_FINETUNE = int(os.environ.get("DEEPFAAKE_EPOCHS_FINETUNE", "16"))
PATIENCE = 4
USE_AMP = os.environ.get("DEEPFAAKE_AMP", "1") == "1" and torch.cuda.is_available()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"torch {torch.__version__} | torchvision | device={device} | AMP={USE_AMP}")
print(f"Dataset  : {DATASET_DIR}")
print(f"CSV      : {SPLIT_CSV}")
print(f"CNN      : {CNN_MODEL_PATH} (présent : {CNN_MODEL_PATH.is_file()})")
print(f"Sorties  : {OUTPUT_DIR}")


In [ ]:
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f"Dossier du dataset absent : {DATASET_DIR}. "
        "Définissez DEEPFAAKE_DATASET_DIR ou modifiez la cellule de configuration."
    )
if not SPLIT_CSV.is_file():
    raise FileNotFoundError(
        f"CSV de split absent : {SPLIT_CSV}. "
        "Définissez DEEPFAAKE_SPLIT_CSV ou modifiez la cellule de configuration."
    )

df = pd.read_csv(SPLIT_CSV)
required = {"rel_path", "label", "video", "split"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Colonnes manquantes dans le CSV : {sorted(missing)}")
if df.empty or df[list(required)].isna().any().any():
    raise ValueError("Le CSV est vide ou contient des valeurs manquantes dans les colonnes requises.")
if not set(df["label"].unique()).issubset({0, 1}):
    raise ValueError("Les labels doivent être encodés 0=REAL et 1=FAKE.")
if set(df["split"].unique()) != {"train", "val", "test"}:
    raise ValueError("Les splits attendus sont exactement train, val et test.")
if df.groupby("video")["split"].nunique().max() != 1:
    raise ValueError("Fuite de données : au moins une vidéo apparaît dans plusieurs splits.")

df["path"] = df["rel_path"].map(lambda rel: DATASET_DIR / rel)
missing_files = [str(path) for path in df["path"] if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} images du CSV sont introuvables. Exemple : {missing_files[0]}"
    )

counts = df.groupby(["split", "label"]).size().unstack(fill_value=0).rename(columns={0: "REAL", 1: "FAKE"})
print("Images par split :")
display(counts)
print("Part de FAKE par split :")
display((counts["FAKE"] / counts.sum(axis=1)).round(4).to_frame("ratio_fake"))
print("Vidéos par split :")
display(df.groupby("split")["video"].nunique().to_frame("videos"))
print(f"Total : {len(df):,} images, {df['video'].nunique():,} vidéos")

# Avertissement (pas une erreur) : une identité peut apparaître dans un vrai et un faux.
train_ids = set(df.loc[df.split == "train", "video"])
test_ids = set(df.loc[df.split == "test", "video"])
print(f"Vidéos de test absentes du train : {len(test_ids - train_ids)} / {len(test_ids)}")


In [ ]:
weights = ResNet18_Weights.DEFAULT
MEAN = tuple(weights.transforms().mean)
STD = tuple(weights.transforms().std)
print(f"Normalisation ImageNet : mean={MEAN} std={STD}")

train_transform = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    # Rotation modérée : au-delà, les artefacts de blending deviennent illisibles.
    v2.RandomRotation(degrees=7),
    v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=MEAN, std=STD),
])
eval_transform = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=MEAN, std=STD),
])


class FaceDataset(Dataset):
    """Lit une image à la demande (pas de dataset complet en RAM)."""

    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            # Interpolation explicite : identique à l'application Flask.
            image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
            array = np.array(image, dtype=np.uint8)
        x = torch.from_numpy(array).permute(2, 0, 1)
        if self.transform is not None:
            x = self.transform(x)
        return x, torch.tensor(float(row.label), dtype=torch.float32)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


train_df = df[df.split == "train"]
val_df = df[df.split == "val"]
test_df = df[df.split == "test"]
if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError("Chaque split doit contenir au moins une image.")

generator = torch.Generator().manual_seed(SEED)
loader_options = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
)
train_loader = DataLoader(
    FaceDataset(train_df, train_transform),
    shuffle=True,
    generator=generator,
    drop_last=len(train_df) > BATCH_SIZE,  # évite un dernier lot de taille 1 (BatchNorm)
    **loader_options,
)
val_loader = DataLoader(FaceDataset(val_df, eval_transform), shuffle=False, **loader_options)
test_loader = DataLoader(FaceDataset(test_df, eval_transform), shuffle=False, **loader_options)
print(f"Train={len(train_df):,} | validation={len(val_df):,} | test={len(test_df):,}")
print(f"Lots : train={len(train_loader)} | val={len(val_loader)} | test={len(test_loader)}")


In [ ]:
def build_model():
    """ResNet-18 ImageNet + tête binaire, même définition qu'à l'inférence."""
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 1))
    return model.to(device)


model = build_model()
criterion = nn.BCEWithLogitsLoss()
checkpoint_path = OUTPUT_DIR / "resnet18_best.pt"
print(model.fc)


In [ ]:
def run_epoch(model, loader, optimizer=None, scaler=None):
    """Une passe complète ; renvoie perte, exactitude et AUC."""
    training = optimizer is not None
    model.train(training)
    loss_sum, probabilities, targets = 0.0, [], []

    with torch.set_grad_enabled(training):
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True).unsqueeze(1)
            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)

            if training:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

            loss_sum += loss.item() * len(y)
            probabilities.extend(torch.sigmoid(logits.detach().float()).cpu().numpy().ravel())
            targets.extend(y.detach().cpu().numpy().ravel())

    probabilities = np.asarray(probabilities)
    targets = np.asarray(targets, dtype=int)
    return {
        "loss": loss_sum / len(loader.dataset),
        "accuracy": accuracy_score(targets, probabilities >= 0.5),
        "auc": roc_auc_score(targets, probabilities) if len(np.unique(targets)) == 2 else float("nan"),
    }


history = []
best_val_auc = -np.inf
epochs_without_improvement = 0


def fit(phase, epochs, optimizer, scheduler=None, scaler=None):
    """Entraîne et sauvegarde le meilleur checkpoint **selon l'AUC de validation**."""
    global best_val_auc, epochs_without_improvement

    for epoch in range(epochs):
        train_metrics = run_epoch(model, train_loader, optimizer, scaler)
        val_metrics = run_epoch(model, val_loader)
        if scheduler is not None:
            scheduler.step(val_metrics["auc"])

        record = {"phase": phase, "epoch": len(history) + 1}
        record.update({f"train_{key}": value for key, value in train_metrics.items()})
        record.update({f"val_{key}": value for key, value in val_metrics.items()})
        history.append(record)

        print(
            f"{phase} {epoch + 1}/{epochs} — train loss={train_metrics['loss']:.4f} "
            f"acc={train_metrics['accuracy']:.4f} | val loss={val_metrics['loss']:.4f} "
            f"acc={val_metrics['accuracy']:.4f} auc={val_metrics['auc']:.4f}"
        )

        improved = np.isfinite(val_metrics["auc"]) and val_metrics["auc"] > best_val_auc
        if improved:
            best_val_auc = val_metrics["auc"]
            epochs_without_improvement = 0
            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "val_auc": float(best_val_auc),
                    "epoch": len(history),
                    "img_size": IMG_SIZE,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print("Arrêt anticipé : l'AUC de validation ne progresse plus.")
                return False
    return True


In [ ]:
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP) if USE_AMP else None

# Phase 1 — tête seule (backbone gelé).
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)
continue_training = fit("head", EPOCHS_HEAD, optimizer, scaler=scaler)

# Phase 2 — fine-tuning complet, taux d'apprentissage différenciés.
if continue_training:
    for parameter in model.parameters():
        parameter.requires_grad = True

    head_parameters = list(model.fc.parameters())
    head_ids = {id(parameter) for parameter in head_parameters}
    backbone_parameters = [p for p in model.parameters() if id(p) not in head_ids]

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_parameters, "lr": 1e-5},
            {"params": head_parameters, "lr": 1e-4},
        ],
        weight_decay=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=1, min_lr=1e-7
    )
    fit("finetune", EPOCHS_FINETUNE, optimizer, scheduler=scheduler, scaler=scaler)

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "resnet18_history.csv", index=False)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Meilleur checkpoint rechargé (AUC val = {checkpoint['val_auc']:.4f}, époque {checkpoint['epoch']}).")


## Évaluation

Le test n'est touché qu'ici, après la sélection du checkpoint sur la validation. Le seuil de
décision est lui aussi choisi sur la validation (indice de Youden), puis appliqué tel quel au
test : aucun réglage ne dépend des résultats de test.

Deux niveaux sont rapportés :

- **frame** : chaque image est un exemple — comparable aux chiffres historiques, mais optimiste
  car les frames d'une même vidéo sont presque identiques ;
- **vidéo** : moyenne des probabilités par vidéo puis une décision par vidéo — c'est le chiffre
  à retenir.


In [ ]:
@torch.inference_mode()
def predict_probabilities(model, loader):
    """Probabilités « FAKE » et labels, dans l'ordre du loader (shuffle=False)."""
    model.eval()
    probabilities, labels = [], []
    for x, y in loader:
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x.to(device, non_blocking=True))
        probabilities.extend(torch.sigmoid(logits.float()).cpu().numpy().ravel())
        labels.extend(y.numpy().ravel())
    return np.asarray(labels, dtype=int), np.asarray(probabilities)


y_val, p_val = predict_probabilities(model, val_loader)
print(f"Validation : {len(y_val)} images, AUC = {roc_auc_score(y_val, p_val):.4f}")


In [ ]:
def best_threshold(y_true, y_prob, criterion="youden"):
    """Seuil optimal sur la validation : Youden (par défaut) ou F1 FAKE."""
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    if len(np.unique(y_true)) < 2:
        return 0.5

    candidates = np.unique(np.concatenate(([0.0], y_prob, [1.0])))
    best_score, best_value = -np.inf, 0.5
    for threshold in candidates:
        predicted = (y_prob >= threshold).astype(int)
        recall_fake = recall_score(y_true, predicted, pos_label=1, zero_division=0)
        recall_real = recall_score(y_true, predicted, pos_label=0, zero_division=0)
        f1_fake = f1_score(y_true, predicted, pos_label=1, zero_division=0)
        score = (recall_fake + recall_real - 1) if criterion == "youden" else f1_fake
        if score > best_score:
            best_score, best_value = score, float(threshold)
    return best_value


THRESHOLD = best_threshold(y_val, p_val, "youden")
print(f"Seuil retenu sur la validation (Youden) : {THRESHOLD:.4f}")
print(f"(seuil par défaut 0.5 : accuracy validation = {accuracy_score(y_val, p_val >= 0.5):.4f})")
print(f"(seuil retenu        : accuracy validation = {accuracy_score(y_val, p_val >= THRESHOLD):.4f})")
print()
print(classification_report(y_val, (p_val >= THRESHOLD).astype(int), labels=[0, 1],
                            target_names=["REAL", "FAKE"], digits=4, zero_division=0))


In [ ]:
def metrics_at(y_true, y_prob, threshold=0.5):
    """Métriques de classification à un seuil donné."""
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    predicted = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, predicted),
        "auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan"),
        "precision_fake": precision_score(y_true, predicted, pos_label=1, zero_division=0),
        "recall_fake": recall_score(y_true, predicted, pos_label=1, zero_division=0),
        "f1_fake": f1_score(y_true, predicted, pos_label=1, zero_division=0),
        "f1_real": f1_score(y_true, predicted, pos_label=0, zero_division=0),
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else float("nan"),
    }


def video_level_metrics(test_frame, y_true, y_prob, threshold=0.5):
    """Agrège par vidéo (moyenne des probabilités) puis calcule les métriques."""
    table = pd.DataFrame(
        {"video": test_frame["video"].values, "label": np.asarray(y_true), "prob": np.asarray(y_prob)}
    )
    if table.groupby("video")["label"].nunique().max() != 1:
        raise ValueError("Une vidéo porte deux labels différents.")
    grouped = table.groupby("video").agg(
        label=("label", "first"), prob=("prob", "mean"), frames=("prob", "size")
    )
    return metrics_at(grouped["label"].values, grouped["prob"].values, threshold), grouped.reset_index()


def bootstrap_ci(metric, y_true, y_score, n_boot=500, alpha=0.05, seed=SEED):
    """IC percentile par bootstrap sur les images."""
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        indices = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            continue
        values.append(metric(y_true[indices], y_score[indices]))
    if not values:
        return (float("nan"), float("nan"))
    return tuple(np.percentile(values, [100 * alpha / 2, 100 * (1 - alpha / 2)]))


def mcnemar(y_true, pred_a, pred_b):
    """Test de McNemar exact bilatéral (paires discordantes)."""
    import math

    y_true = np.asarray(y_true, dtype=int)
    correct_a = np.asarray(pred_a, dtype=int) == y_true
    correct_b = np.asarray(pred_b, dtype=int) == y_true
    a_only = int((correct_a & ~correct_b).sum())
    b_only = int((~correct_a & correct_b).sum())
    discordant = a_only + b_only
    if discordant == 0:
        return a_only, b_only, 1.0
    tail = sum(math.comb(discordant, k) for k in range(min(a_only, b_only) + 1))
    return a_only, b_only, min(1.0, 2.0 * tail / 2.0**discordant)


In [ ]:
y_true, y_prob = predict_probabilities(model, test_loader)
frame_metrics = metrics_at(y_true, y_prob, THRESHOLD)
video_metrics, video_table = video_level_metrics(test_df, y_true, y_prob, THRESHOLD)

auc_low, auc_high = bootstrap_ci(roc_auc_score, y_true, y_prob)
acc_low, acc_high = bootstrap_ci(lambda yt, ys: accuracy_score(yt, ys >= THRESHOLD), y_true, y_prob)

summary = pd.DataFrame({"frame": frame_metrics, "video": video_metrics})
display(summary.style.format("{:.4f}"))

print(f"AUC test (frame) : {frame_metrics['auc']:.4f}  IC95% [{auc_low:.4f}, {auc_high:.4f}]")
print(f"Accuracy test    : {frame_metrics['accuracy']:.4f}  IC95% [{acc_low:.4f}, {acc_high:.4f}]")
print(f"Vidéos évaluées  : {len(video_table):,}  (frames/vidéo médianes : {video_table['frames'].median():.0f})")
print()
print("=== Rapport détaillé (frame) ===")
print(classification_report(y_true, (y_prob >= THRESHOLD).astype(int), labels=[0, 1],
                            target_names=["REAL", "FAKE"], digits=4, zero_division=0))


In [ ]:
# --- Courbes d'apprentissage -------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, metric in zip(axes, ["loss", "accuracy", "auc"]):
    history_df.plot(x="epoch", y=[f"train_{metric}", f"val_{metric}"], ax=axis, title=metric.capitalize())
    head_epochs = int((history_df["phase"] == "head").sum())
    axis.axvline(head_epochs + 0.5, color="grey", linestyle="--", linewidth=1, alpha=0.7)
    axis.set_xlabel("Époque (globale)")
    axis.grid(alpha=0.2)
axes[0].set_title("Perte")
axes[1].set_title("Exactitude")
axes[2].set_title("AUC")
fig.suptitle("ResNet-18 — la ligne pointillée sépare la tête du fine-tuning", fontsize=10)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "resnet18_courbes.png", dpi=150, bbox_inches="tight")
plt.show()

# --- ROC / précision-rappel --------------------------------------------
fpr, tpr, _ = roc_curve(y_true, y_prob)
precision, recall, _ = precision_recall_curve(y_true, y_prob)
predicted = (y_prob >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
tpr_point = tp / (tp + fn) if (tp + fn) else 0.0
fpr_point = fp / (fp + tn) if (fp + tn) else 0.0

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, label=f"ResNet-18 (AUC = {frame_metrics['auc']:.4f})")
axes[0].plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
axes[0].scatter([fpr_point], [tpr_point], color="tab:red", zorder=5, label=f"seuil {THRESHOLD:.3f}")
axes[0].set_xlabel("Taux de faux positifs (REAL classés FAKE)")
axes[0].set_ylabel("Taux de vrais positifs (FAKE détectés)")
axes[0].set_title("Courbe ROC — test")
axes[0].legend(loc="lower right", fontsize=9)
axes[0].grid(alpha=0.2)

axes[1].plot(recall, precision, label="ResNet-18")
axes[1].axhline(y_true.mean(), color="grey", linestyle="--", linewidth=1, label="hasard")
axes[1].set_xlabel("Rappel (FAKE)")
axes[1].set_ylabel("Précision (FAKE)")
axes[1].set_title("Courbe précision-rappel — test")
axes[1].legend(loc="lower left", fontsize=9)
axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "resnet18_roc_pr.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Matrice de confusion ----------------------------------------------
fig, axis = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay(confusion_matrix(y_true, predicted, labels=[0, 1]),
                       display_labels=["REAL", "FAKE"]).plot(ax=axis, colorbar=False, cmap="Greens")
axis.set_title(f"ResNet-18 — test (seuil {THRESHOLD:.3f})")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "resnet18_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Diagramme de fiabilité --------------------------------------------
bins = np.linspace(0, 1, 11)
indices = np.clip(np.digitize(y_prob, bins) - 1, 0, 9)
confidence, frequency, counts = [], [], []
for index in range(10):
    mask = indices == index
    if mask.any():
        confidence.append(float(y_prob[mask].mean()))
        frequency.append(float(y_true[mask].mean()))
        counts.append(int(mask.sum()))
ece = float(np.sum(np.abs(np.array(confidence) - np.array(frequency)) * np.array(counts)) / len(y_true))

fig, axis = plt.subplots(figsize=(4.8, 4.2))
axis.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="calibration parfaite")
axis.plot(confidence, frequency, marker="o", label=f"ResNet-18 (ECE = {ece:.3f})")
axis.set_xlabel("Probabilité prédite (FAKE)")
axis.set_ylabel("Fréquence observée de FAKE")
axis.set_title("Diagramme de fiabilité — test")
axis.legend(fontsize=9)
axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "resnet18_calibration.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"ECE = {ece:.4f} (0 = parfaitement calibré ; au-dessus de ~0.1, les probabilités sont trompeuses)")


In [ ]:
# Les couches d'augmentation sont dans le graphe du .keras : sans ce nettoyage,
# la même image donnerait deux probabilités différentes (vérifié : 6/20 verdicts instables).
RANDOM_LAYERS = ("RandomFlip", "RandomRotation", "RandomZoom", "RandomTranslation",
                 "RandomBrightness", "RandomContrast", "RandomCrop", "RandomErasing",
                 "Dropout", "SpatialDropout2D", "GaussianNoise", "GaussianDropout")


def strip_random_layers(keras_model):
    """Reconstruit un graphe d'inférence déterministe avec les mêmes poids."""
    from keras import layers as L
    from keras import Model, Input

    removable = tuple(getattr(L, name) for name in RANDOM_LAYERS if hasattr(L, name))
    inputs = Input(shape=keras_model.input_shape[1:])
    outputs, removed = inputs, []
    for layer in keras_model.layers[1:]:
        if isinstance(layer, removable):
            removed.append(layer.__class__.__name__)
            continue
        outputs = layer(outputs)
    return Model(inputs, outputs), removed


def predict_cnn(cnn_model, frame):
    """Probabilités du CNN par lots, sans charger tout le split en RAM."""
    paths = frame["path"].tolist()
    probabilities = []
    for start in range(0, len(paths), BATCH_SIZE):
        images = []
        for path in paths[start:start + BATCH_SIZE]:
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image).convert("RGB")
                image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
                images.append(np.asarray(image, dtype=np.uint8))
        probabilities.extend(cnn_model.predict(np.stack(images), verbose=0).ravel())
    return np.asarray(probabilities)


cnn_probs = cnn_probs_val = None
cnn_threshold = None
try:
    import tensorflow as tf

    if not CNN_MODEL_PATH.is_file():
        raise FileNotFoundError(f"{CNN_MODEL_PATH} absent")

    raw_cnn = tf.keras.models.load_model(CNN_MODEL_PATH)
    cnn_model, removed = strip_random_layers(raw_cnn)
    print(f"CNN chargé — couches stochastiques retirées : {', '.join(removed) or 'aucune'}")

    probe = np.zeros((4, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
    first, second = cnn_model.predict(probe, verbose=0), cnn_model.predict(probe, verbose=0)
    if not np.allclose(first, second):
        raise RuntimeError("le CNN reste non déterministe après nettoyage")
    print("Test de déterminisme : OK (deux appels identiques → mêmes sorties)")

    cnn_probs_val = predict_cnn(cnn_model, val_df)
    cnn_probs = predict_cnn(cnn_model, test_df)
    cnn_threshold = best_threshold(y_val, cnn_probs_val, "youden")
    cnn_frame = metrics_at(y_true, cnn_probs, cnn_threshold)
    print(f"Seuil CNN (validation) : {cnn_threshold:.4f}")

    comparison = pd.DataFrame(
        {
            "ResNet-18": frame_metrics,
            "CNN Simple (déterministe)": cnn_frame,
        }
    )
    display(comparison.style.format("{:.4f}"))

    a_only, b_only, p_value = mcnemar(
        y_true,
        (y_prob >= THRESHOLD).astype(int),
        (cnn_probs >= cnn_threshold).astype(int),
    )
    print(f"McNemar ResNet vs CNN : p = {p_value:.4g} "
          f"(ResNet seul correct : {a_only}, CNN seul correct : {b_only})")
    if p_value >= 0.05:
        print("→ écart non significatif au seuil de 5 % : ne pas conclure à une supériorité.")
    else:
        print("→ écart significatif au seuil de 5 %.")

    cnn_model.save(OUTPUT_DIR / "cnn_simple_inference.keras")
    print(f"Modèle d'inférence propre écrit : {OUTPUT_DIR / 'cnn_simple_inference.keras'}")
except Exception as exc:  # noqa: BLE001 - la comparaison est facultative
    print(f"Comparaison CNN ignorée : {type(exc).__name__}: {exc}")


In [ ]:
predictions = pd.DataFrame(
    {
        "rel_path": test_df["rel_path"].values,
        "video": test_df["video"].values,
        "label": y_true,
        "prob_resnet": y_prob,
        "pred_resnet": (y_prob >= THRESHOLD).astype(int),
        "prob_cnn": cnn_probs if cnn_probs is not None else np.nan,
    }
)
predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
print(f"Prédictions écrites : {OUTPUT_DIR / 'predictions.csv'}")

# --- Grille des erreurs les plus confiantes -----------------------------
wrong = np.where(predictions["pred_resnet"].values != y_true)[0]
confidence = np.abs(y_prob - THRESHOLD)
order = wrong[np.argsort(-confidence[wrong])][:12] if len(wrong) else []

if len(order):
    columns = 4
    rows = int(np.ceil(len(order) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 2.6, rows * 3.0))
    axes = np.atleast_1d(axes).ravel()
    for axis in axes:
        axis.axis("off")
    for axis, index in zip(axes, order):
        with Image.open(test_df.iloc[index]["path"]) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"vrai={y_true[index]} · prédit={predictions['pred_resnet'].iloc[index]}\n"
            f"p(FAKE)={y_prob[index]:.3f}",
            fontsize=8,
        )
    fig.suptitle(f"12 erreurs les plus confiantes sur {len(wrong):,} erreurs", fontsize=10)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "resnet18_erreurs.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Aucune erreur de classification sur ce jeu de test.")


In [ ]:
class GradCAM:
    """Carte d'activation (Grad-CAM) sur la dernière couche convolutive."""

    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._handles = [
            target_layer.register_forward_hook(self._forward),
            target_layer.register_full_backward_hook(self._backward),
        ]

    def _forward(self, _module, _inputs, output):
        self.activations = output.detach()

    def _backward(self, _module, _grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, tensor):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(tensor)
        logits.sum().backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).clamp(min=0).squeeze()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy()

    def close(self):
        for handle in self._handles:
            handle.remove()


def show_gradcam(indices, filename):
    cam = GradCAM(model, model.layer4[-1])
    fig, axes = plt.subplots(1, len(indices), figsize=(3.2 * len(indices), 3.4))
    axes = np.atleast_1d(axes)
    try:
        for axis, index in zip(axes, indices):
            with Image.open(test_df.iloc[index]["path"]) as image:
                face = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
            tensor = eval_transform(torch.from_numpy(np.array(face)).permute(2, 0, 1)).unsqueeze(0).to(device)
            with torch.enable_grad():
                heatmap = cam(tensor)
            axis.imshow(face)
            axis.imshow(heatmap, cmap="jet", alpha=0.45)
            axis.set_title(
                f"vrai={y_true[index]} · p(FAKE)={y_prob[index]:.3f}", fontsize=9
            )
            axis.axis("off")
    finally:
        cam.close()
    fig.suptitle("Grad-CAM : zones qui poussent la décision vers FAKE", fontsize=10)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


# Deux vrais positifs et deux faux négatifs (à adapter si le test est petit).
tp_indices = np.where((predictions["pred_resnet"].values == 1) & (y_true == 1))[0][:2]
fn_indices = np.where((predictions["pred_resnet"].values == 0) & (y_true == 1))[0][:2]
sample = list(tp_indices) + list(fn_indices)
if sample:
    show_gradcam(sample, "resnet18_gradcam.png")
else:
    print("Pas d'exemple FAKE disponible pour la Grad-CAM.")


In [ ]:
import io


def jpeg_recompress(image, quality):
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)
    with Image.open(buffer) as compressed:
        return compressed.convert("RGB")


def predict_paths(paths, quality=None):
    """Probabilités ResNet sur une liste de chemins, avec recompression optionnelle."""
    probabilities = []
    for start in range(0, len(paths), BATCH_SIZE):
        tensors = []
        for path in paths[start:start + BATCH_SIZE]:
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image).convert("RGB")
                image = image.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
                if quality is not None:
                    image = jpeg_recompress(image, quality)
                tensors.append(eval_transform(torch.from_numpy(np.array(image)).permute(2, 0, 1)))
        with torch.inference_mode():
            logits = model(torch.stack(tensors).to(device))
        probabilities.extend(torch.sigmoid(logits.float()).cpu().numpy().ravel())
    return np.asarray(probabilities)


# Sous-ensemble borné : le test complet doublerait le temps d'évaluation.
subset_size = min(2000, len(test_df))
rng = np.random.default_rng(SEED)
subset = np.sort(rng.choice(len(test_df), size=subset_size, replace=False))
robust_paths = [test_df.iloc[index]["path"] for index in subset]
robust_labels = y_true[subset]

rows = {"original": metrics_at(robust_labels, predict_paths(robust_paths), THRESHOLD)}
for quality in (90, 70, 50):
    rows[f"JPEG q={quality}"] = metrics_at(
        robust_labels, predict_paths(robust_paths, quality=quality), THRESHOLD
    )

robust_df = pd.DataFrame(rows).T
display(robust_df.style.format("{:.4f}"))
print(f"Sous-ensemble : {subset_size} images du test.")
print("Un détecteur utile doit conserver l'essentiel de son AUC sous recompression JPEG,")
print("car les vidéos réelles circulent presque toujours recompressées.")


In [ ]:
artifacts = {
    "history": OUTPUT_DIR / "resnet18_history.csv",
    "predictions": OUTPUT_DIR / "predictions.csv",
    "courbes": OUTPUT_DIR / "resnet18_courbes.png",
    "roc_pr": OUTPUT_DIR / "resnet18_roc_pr.png",
    "confusion": OUTPUT_DIR / "resnet18_confusion.png",
    "calibration": OUTPUT_DIR / "resnet18_calibration.png",
}

card_path = PROJECT_DIR / "model_card.json"
card = json.loads(card_path.read_text(encoding="utf-8")) if card_path.is_file() else {}
card.update(
    {
        "img_size": IMG_SIZE,
        "decision_threshold": THRESHOLD,
        "seed": SEED,
        "trained_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "metrics": {
            "status": "reproductible",
            "frame": frame_metrics,
            "video": video_metrics,
            "auc_ci95": [auc_low, auc_high],
            "ece": ece,
            "cnn_deterministic": cnn_probs is not None,
            "cnn_frame": (
                metrics_at(y_true, cnn_probs, cnn_threshold)
                if cnn_probs is not None and cnn_threshold is not None
                else None
            ),
            "test_videos": int(len(video_table)),
        },
    }
)
(OUTPUT_DIR / "model_card.json").write_text(json.dumps(card, indent=2, ensure_ascii=False), encoding="utf-8")

print("Artefacts :")
for name, path in artifacts.items():
    print(f"  {name:12s} {'OK ' if path.is_file() else '—  '} {path}")
print(f"  carte de modèle  OK  {OUTPUT_DIR / 'model_card.json'}")
print()
print("Pour déployer ces résultats dans l'application :")
print(f"  1. copier {OUTPUT_DIR / 'model_card.json'} vers {card_path}")
print(f"  2. remplacer cnn_simple.keras par {OUTPUT_DIR / 'cnn_simple_inference.keras'} (si présent)")
print("  3. relancer app.py — le seuil de décision sera lu depuis model_card.json")


## Interprétation et limites

- Une image analysée est un **crop de visage** : le modèle ne fournit pas, à lui seul, la preuve
  qu'une vidéo entière est truquée. Les métriques vidéo de ce notebook sont une agrégation des
  scores de frames, pas une analyse temporelle (pas de cohérence inter-frames, pas d'audio).
- La séparation par vidéo empêche la fuite directe, mais **pas la fuite d'identité** : une même
  personne peut apparaître dans une vidéo réelle d'entraînement et dans une vidéo synthétique de
  test. Une séparation par identité serait plus stricte ; à mentionner comme limite.
- Celeb-DF v2 contient une seule famille de manipulations (FaceSwap). La généralisation à
  d'autres générateurs (diffusion, avatars temps réel) ou à des vidéos compressées n'est pas
  garantie : le test de robustesse JPEG en donne une première indication, pas une preuve.
- Le niveau frame surestime systématiquement la performance : c'est pourquoi les deux niveaux
  sont rapportés, et pourquoi seul le niveau vidéo doit être présenté comme résultat principal.
- Un seuil de décision unique n'est pas forcément pertinent selon l'usage (contrôle qualité,
  modération, expertise forensique) : la courbe précision-rappel permet de choisir selon le coût
  relatif des faux positifs et des faux négatifs.
- Les probabilités ne sont pas garanties calibrées (voir l'ECE) : les utiliser comme « confiance »
  brute dans une interface est optimiste, surtout loin de 0 et 1.
